**Feature pruning** is the model-free, post-hoc reduction of a ``df_feat`` (here from :meth:`CPP.run` on the gamma-secretase ``DOM_GSEC`` dataset) before model-based selection. :meth:`SequenceFeature.prune_by_variance` drops near-constant features — those whose feature values barely change across the samples — measured on the feature matrix that :meth:`SequenceFeature.feature_matrix` builds from ``df_parts``. Recommended order: **variance -> correlation -> select_features**.

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False

df_seq = aa.load_dataset(name="DOM_GSEC", n=20)
labels = df_seq["label"].to_list()
sf = aa.SequenceFeature()
# gamma-secretase geometry: the TMD (~20 aa) comes from each protein's tmd_start/tmd_stop,
# flanked by short juxtamembrane domains of 4 residues each.
df_parts = sf.get_df_parts(df_seq=df_seq, jmd_n_len=4, jmd_c_len=4)
df_feat = aa.CPP(df_parts=df_parts).run(labels=labels, n_filter=50)
print(f"features from CPP.run: {len(df_feat)}")

features from CPP.run: 50


With the default ``threshold=0.0`` only strictly constant features are removed (population variance over all samples; ``threshold`` is in variance units). The result is a row-filtered ``df_feat`` with the same schema:

In [2]:
df_var = sf.prune_by_variance(df_feat=df_feat, df_parts=df_parts, threshold=0.0)
print(f"kept {len(df_var)} of {len(df_feat)} features")
df_var.head()

kept 50 of 50 features


,feature,category,subcategory,scale_name,scale_description,abs_auc,abs_mean_dif,mean_dif,std_test,std_ref,p_val_mann_whitney,p_val_fdr_bh,positions
0,"TMD-Pattern(C,4,8)-BEGF750101",Conformation,α-helix,α-helix,Conformational parameter of inner helix (Beghi...,0.444,0.299,0.299,0.090,0.196,0.000002,0.022853,"23,27"
1,"TMD_C_JMD_C-Pattern(N,4,8)-BEGF750101",Conformation,α-helix,α-helix,Conformational parameter of inner helix (Beghi...,0.444,0.299,0.299,0.090,0.196,0.000002,0.045706,"24,28"
2,"TMD_C_JMD_C-Pattern(C,4,8,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized frequency of turn (Crawford et al.,...",0.431,0.251,-0.251,0.088,0.143,0.000003,0.006359,"29,33,37"
3,"TMD_C_JMD_C-Pattern(N,4,8,12)-CRAJ730103",Conformation,β-turn,β-turn,"Normalized frequency of turn (Crawford et al.,...",0.431,0.251,-0.251,0.088,0.143,0.000003,0.008903,"24,28,32"
4,"TMD_C_JMD_C-Pattern(C,4,8,12)-MUNV940102",Energy,Free energy (folding),Free energy (α-helix),Free energy in alpha-helical region (Munoz-Ser...,0.422,0.148,-0.148,0.056,0.097,0.000005,0.004132,"29,33,37"


The remaining parameters control how the feature matrix is built or let you reuse one. A custom ``df_scales`` overrides the default scale set; ``accept_gaps`` tolerates gapped parts; ``n_jobs`` parallelizes the build. A pre-computed ``X`` (column ``i`` = feature in row ``i``) skips the build entirely and also covers a :meth:`CPP.run_num` ``df_feat``:

In [3]:
df_scales = aa.load_scales()
df_var_full = sf.prune_by_variance(df_feat=df_feat, df_parts=df_parts, df_scales=df_scales,
                                   threshold=0.005, accept_gaps=True, n_jobs=1)
X = sf.feature_matrix(features=df_feat, df_parts=df_parts)
df_var_X = sf.prune_by_variance(df_feat=df_feat, X=X, threshold=0.005)
print(f"via df_parts+params: {len(df_var_full)} | via pre-computed X: {len(df_var_X)}")

via df_parts+params: 50 | via pre-computed X: 50


**What can go wrong?** A ``threshold`` above every feature's variance would remove all features, so it raises a ``ValueError`` rather than returning an empty table:

In [4]:
try:
    sf.prune_by_variance(df_feat=df_feat, X=X, threshold=1e6)
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: 'threshold' (1000000.0) removed all features. Lower it to retain features.
